# Data Cleaning Pipeline

This notebook cleans the MXMH survey data.

In [16]:
import pandas as pd
import numpy as np
from pathlib import Path

In [17]:
# ----------------------------
# Settings
# ----------------------------
PREFERRED_FILES = [
    "mxmh_survey_results.csv",
]

OUTPUT_FILE = "mxmh_clean.csv"
TARGET_COL = "anxiety"   # options: anxiety, depression, insomnia, ocd
HIGH_THRESHOLD = 7       # for classification target

In [18]:
# ----------------------------
# Helpers
# ----------------------------
def find_data_file() -> Path:
    here = Path.cwd()  # Look in the current working directory
    for name in PREFERRED_FILES:
        p = here / name
        if p.exists():
            return p

    # fallback: pick first csv/xlsx in folder
    candidates = list(here.glob("*.csv")) + list(here.glob("*.xlsx")) + list(here.glob("*.xls"))
    if candidates:
        return candidates[0]

    raise FileNotFoundError(
        f"No dataset found in {here}. Expected one of {PREFERRED_FILES} "
        f"or any .csv/.xlsx file in the current directory."
    )

def read_table(path: Path) -> pd.DataFrame:
    if path.suffix.lower() in [".xlsx", ".xls"]:
        return pd.read_excel(path)

    # CSV: try normal read, then common fallbacks
    try:
        return pd.read_csv(path)
    except UnicodeDecodeError:
        return pd.read_csv(path, encoding="latin-1")
    except pd.errors.ParserError:
        # try semicolon delimiter, then python engine
        try:
            return pd.read_csv(path, sep=";")
        except Exception:
            return pd.read_csv(path, engine="python")

def clean_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = (
        df.columns.astype(str)
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
        .str.replace("/", "_")
        .str.replace("-", "_")
        .str.replace(r"[^\w_]", "", regex=True)
    )
    return df

In [19]:
# ----------------------------
# Load
# ----------------------------
data_path = find_data_file()
print(f"Working directory: {Path.cwd()}")
print(f"Using dataset file: {data_path.name}")

df = read_table(data_path)
print("Loaded shape:", df.shape)

Working directory: c:\machine-learning-course\data\final assessment
Using dataset file: mxmh_survey_results.csv
Loaded shape: (736, 33)


In [20]:
# ----------------------------
# Clean
# ----------------------------
df = clean_columns(df)

# Drop non-modelling columns if present
df = df.drop(columns=["timestamp", "permissions"], errors="ignore")

# Coerce key numeric columns if present
for col in ["age", "hours_per_day", "bpm", "anxiety", "depression", "insomnia", "ocd"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# Remove unrealistic ages
if "age" in df.columns:
    df = df[(df["age"] >= 10) & (df["age"] <= 100)]

# Clip listening hours
if "hours_per_day" in df.columns:
    df["hours_per_day"] = df["hours_per_day"].clip(lower=0, upper=24)

# Impute missing values
numeric_cols = df.select_dtypes(include=["number"]).columns.tolist()
cat_cols = df.select_dtypes(include=["object", "bool"]).columns.tolist()

for col in numeric_cols:
    if df[col].isna().any():
        df[col] = df[col].fillna(df[col].median())

for col in cat_cols:
    if df[col].isna().any():
        df[col] = df[col].fillna(df[col].mode(dropna=True)[0])

# Map common Yes/No columns if they exist
yes_no_cols = ["while_working", "instrumentalist", "composer", "exploratory", "foreign_languages"]
for col in yes_no_cols:
    if col in df.columns:
        df[col] = df[col].map({"Yes": 1, "No": 0, "yes": 1, "no": 0}).fillna(df[col])

# Map frequency columns if present
freq_mapping = {"Never": 0, "Rarely": 1, "Sometimes": 2, "Very frequently": 3}
for col in df.columns:
    if "frequency" in col:
        df[col] = df[col].map(freq_mapping).fillna(df[col])

# Create classification target
if TARGET_COL in df.columns:
    df[f"high_{TARGET_COL}"] = (df[TARGET_COL] >= HIGH_THRESHOLD).astype(int)
else:
    print(f"Target column '{TARGET_COL}' not found. Available columns:")
    print(sorted(df.columns))

In [21]:
df = df.dropna()

print("Cleaned shape:", df.shape)
print("Top missing values after cleaning:")
print(df.isna().sum().sort_values(ascending=False).head(10))

df.to_csv("mxmh_clean_new.csv", index=False)
print(f"Saved cleaned file: mxmh_clean_new.csv")

Cleaned shape: (735, 32)
Top missing values after cleaning:
age                           0
primary_streaming_service     0
music_effects                 0
ocd                           0
insomnia                      0
depression                    0
anxiety                       0
frequency_video_game_music    0
frequency_rock                0
frequency_rap                 0
dtype: int64
Saved cleaned file: mxmh_clean_new.csv
